# 📦 Master Product Dimension (`dim_products`) Lakehouse Pipeline
This notebook implements the complete end-to-end Medallion pipeline for the **Product Master Dimension**:
* **Bronze:** Ingests raw CSV catalogs with explicit schemas and Change Data Feed.
* **Silver:** Deduplicates `product_id`, cleans category casing, fixes typos, maps corporate divisions, and computes deterministic surrogate key `product_code`.
* **Gold:** Persists subsidiary dimension `sb_dim_products` and executes SCD Type 1 merge into enterprise parent dimension `dim_products`, finalized with Delta Lake Z-ORDER optimization.

### 📌 Step 1: Import Core PySpark & Delta Lake Libraries
* **Purpose:** Loads necessary PySpark SQL analytical functions and Delta Lake table abstractions required for data manipulation and Lakehouse ACID merges.
* **Logic & Transformations:** Imports `pyspark.sql.functions as F` and `DeltaTable` from `delta.tables`.
* **Inputs & Dependencies:** PySpark runtime and `delta-spark` package.
* **Outputs & Medallion State:** Module namespaces `F` and `DeltaTable` available in session scope.

In [3]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### 📌 Step 2: Runtime Bootstrap & Project Utilities Execution
* **Purpose:** Configures modular Python paths and executes shared project utilities to establish active environment configurations, conformed schemas, and audit tools.
* **Logic & Transformations:** Resolves repository root path on `sys.path`, executes `%run ./utilities`, and initializes Databricks compatibility shims.
* **Inputs & Dependencies:** Shared Lakehouse utilities (`./utilities.py` / `utilities.ipynb`).
* **Outputs & Medallion State:** Pre-populated `spark`, `dbutils`, `display`, and global configurations in session scope.

In [4]:
# Initialize environment & Databricks compatibility (noop in Databricks)
import sys, os
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, "..")) if os.path.basename(current_dir) in ["1_setup", "2_dimension_data_processing", "3_fact_dat_processing"] else current_dir
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.compat import init_notebook_context
spark, dbutils, display = init_notebook_context(globals())

# Load environment config, schemas, and utilities via relative path
%run ../1_setup/utilities


26/09/17 12:40:59 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### 📌 Step 3: Verify Active Medallion Schema Configurations
* **Purpose:** Confirms that the target Lakehouse schemas (`bronze`, `silver`, `gold`) are properly defined and aligned with the active environment.
* **Logic & Transformations:** Prints `bronze_schema`, `silver_schema`, and `gold_schema` strings to standard output for visual verification.
* **Inputs & Dependencies:** Configuration variables exported by utilities in Step 2.
* **Outputs & Medallion State:** Schema names printed to cell output.

In [5]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


### 📌 Step 4: Pipeline Parameterization via Interactive Widgets
* **Purpose:** Establishes configurable parameters (`catalog`, `data_source`) and derives cloud storage URIs for the raw products landing zone.
* **Logic & Transformations:** Defines interactive Databricks text widgets, resolves active catalog and dataset name, and forms S3 paths (`base_path`, `landing_path`, `processed_path`).
* **Inputs & Dependencies:** Databricks widget inputs (`catalog`: `fmcg`, `data_source`: `products`).
* **Outputs & Medallion State:** Pipeline URI paths and target table names (`bronze_table`, `silver_table`, `gold_table`) registered.

In [6]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://spartsbar-2355/{data_source}/*.csv'
print(base_path)

s3://spartsbar-2355/products/*.csv


### 📌 Step 5: Schema-Enforced Ingestion from AWS S3 Landing Zone
* **Purpose:** Ingests raw CSV product catalog files using explicit `StructType` schema enforcement, preventing silent schema drift and costly full-table schema inference.
* **Logic & Transformations:** Reads CSV files with `header=True`, binds explicit `products_schema`, appends `current_timestamp()` as `read_timestamp`, and unpacks `_metadata.file_name` and `_metadata.file_size`.
* **Inputs & Dependencies:** Raw CSV files at `s3://spartsbar-2355/products/*.csv`.
* **Outputs & Medallion State:** Raw DataFrame `df` containing source product data with ingestion audit metadata.

In [7]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(products_schema)  # Explicit schema prevents redundant scan and schema drift
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(10))


26/09/17 12:41:00 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3://spartsbar-2355/products/*.csv.
org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3586)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3617)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3721)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3672)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:558)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:373)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:57)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(Res

[Local Spark Emulation] S3 path detected without AWS credentials. Providing mock data for: s3://spartsbar-2355/products/*.csv


+----------+---------------------------+---------+-----------------------------------------------------------+--------------------------+---------------+---------+
|product_id|product_name               |category |_metadata                                                  |read_timestamp            |file_name      |file_size|
+----------+---------------------------+---------+-----------------------------------------------------------+--------------------------+---------------+---------+
|P101      |Atlikon Cola 500ml         |Beverages|{sample_data.csv, 1024, s3://spartsbar-2355/products/*.csv}|2026-09-17 12:41:00.336377|sample_data.csv|1024     |
|P102      |Atlikon Orange Juice 1L    |Beverages|{sample_data.csv, 1024, s3://spartsbar-2355/products/*.csv}|2026-09-17 12:41:00.336377|sample_data.csv|1024     |
|P103      |Atlikon Potato Chips 150g  |Snacks   |{sample_data.csv, 1024, s3://spartsbar-2355/products/*.csv}|2026-09-17 12:41:00.336377|sample_data.csv|1024     |
|P104      |Atli

### 📌 Step 6: Validate Raw Ingestion Schema
* **Purpose:** Inspects field data types and nullability constraints to ensure strict alignment with enterprise ingestion standards.
* **Logic & Transformations:** Calls `df.printSchema()` to print the DataFrame structural tree.
* **Inputs & Dependencies:** Ingested DataFrame `df` from Step 5.
* **Outputs & Medallion State:** Schema definition logged to cell output.

In [8]:
# print check data type
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- _metadata: struct (nullable = false)
 |    |-- file_name: string (nullable = false)
 |    |-- file_size: long (nullable = false)
 |    |-- file_path: string (nullable = false)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



### 📌 Step 7: Inspect Sample Raw Product Records
* **Purpose:** Visually validates the integrity and layout of landed product records before committing to the Delta Lake storage layer.
* **Logic & Transformations:** Limits DataFrame to 10 rows and invokes `display(df.limit(10))` for tabular presentation.
* **Inputs & Dependencies:** Raw DataFrame `df`.
* **Outputs & Medallion State:** Formatted tabular view rendered in notebook output.

In [9]:
display(df.limit(10))

+----------+---------------------------+---------+-----------------------------------------------------------+--------------------------+---------------+---------+
|product_id|product_name               |category |_metadata                                                  |read_timestamp            |file_name      |file_size|
+----------+---------------------------+---------+-----------------------------------------------------------+--------------------------+---------------+---------+
|P101      |Atlikon Cola 500ml         |Beverages|{sample_data.csv, 1024, s3://spartsbar-2355/products/*.csv}|2026-09-17 12:41:01.491688|sample_data.csv|1024     |
|P102      |Atlikon Orange Juice 1L    |Beverages|{sample_data.csv, 1024, s3://spartsbar-2355/products/*.csv}|2026-09-17 12:41:01.491688|sample_data.csv|1024     |
|P103      |Atlikon Potato Chips 150g  |Snacks   |{sample_data.csv, 1024, s3://spartsbar-2355/products/*.csv}|2026-09-17 12:41:01.491688|sample_data.csv|1024     |
|P104      |Atli

### 📌 Step 8: Persist Raw Ingestion into Bronze Delta Table
* **Purpose:** Saves the landed product data into the immutable Bronze Delta Lake table with Change Data Feed (CDF) enabled for downstream CDC auditing.
* **Logic & Transformations:** Writes `df` with `format('delta')`, sets `delta.enableChangeDataFeed = true`, and saves in `overwrite` mode to `{catalog}.bronze.products`.
* **Inputs & Dependencies:** Ingested DataFrame `df`.
* **Outputs & Medallion State:** Bronze Delta table `fmcg.bronze.products` committed on Delta Lake storage with change data logging.

In [10]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

[Local Spark Emulation] Adapted target table: fmcg.bronze.products -> bronze.products


26/09/17 12:41:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### 📌 Step 9: Query Bronze Table to Initialize Silver Cleansing
* **Purpose:** Reads freshly persisted Bronze records from Delta Lake to isolate raw landing from transformation and conformance logic.
* **Logic & Transformations:** Executes `spark.sql(SELECT * FROM {catalog}.{bronze_schema}.{data_source})` and previews the first 10 rows.
* **Inputs & Dependencies:** Bronze Delta table `fmcg.bronze.products`.
* **Outputs & Medallion State:** DataFrame `df_bronze` loaded into memory for Silver tier processing.

In [11]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

[Local Spark Emulation] Multi-part namespace adapted: SELECT * FROM fmcg.bronze.products; -> SELECT * FROM bronze.products;


+----------+--------------------+---------+--------------------+--------------------+---------------+---------+
|product_id|        product_name| category|           _metadata|      read_timestamp|      file_name|file_size|
+----------+--------------------+---------+--------------------+--------------------+---------------+---------+
|      P101|  Atlikon Cola 500ml|Beverages|{sample_data.csv,...|2026-09-17 12:41:...|sample_data.csv|     1024|
|      P102|Atlikon Orange Ju...|Beverages|{sample_data.csv,...|2026-09-17 12:41:...|sample_data.csv|     1024|
|      P103|Atlikon Potato Ch...|   Snacks|{sample_data.csv,...|2026-09-17 12:41:...|sample_data.csv|     1024|
|      P104|Atlikon Dark Choc...|   Snacks|{sample_data.csv,...|2026-09-17 12:41:...|sample_data.csv|     1024|
+----------+--------------------+---------+--------------------+--------------------+---------------+---------+



### 📌 Step 10: Deduplicate Records by Primary Natural Key
* **Purpose:** Eliminates duplicate product catalog records based on natural identifier `product_id`, reporting row volume before and after deduplication.
* **Logic & Transformations:** Evaluates `df_bronze.count()`, applies `dropDuplicates(['product_id'])`, and logs retained row count.
* **Inputs & Dependencies:** Bronze DataFrame `df_bronze`.
* **Outputs & Medallion State:** Deduplicated DataFrame `df_silver` with duplicate product entries pruned.

In [12]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['product_id'])
print('Rows after duplicates dropped: ', df_silver.count())

Rows before duplicates dropped:  4
Rows after duplicates dropped:  4


### 📌 Step 11: Profile Raw Categorical Values
* **Purpose:** Identifies inconsistent casing, trailing whitespace, or formatting anomalies across product categories.
* **Logic & Transformations:** Queries `df_silver.select('category').distinct().show()` to output distinct raw category values.
* **Inputs & Dependencies:** Deduplicated DataFrame `df_silver`.
* **Outputs & Medallion State:** Terminal tabular listing of distinct category strings.

In [13]:
df_silver.select('category').distinct().show()

+---------+
| category|
+---------+
|Beverages|
|   Snacks|
+---------+



### 📌 Step 12: Standardize Category Casing via Title Case Normalization
* **Purpose:** Normalizes category names into consistent Title Case format, eliminating discrepancies caused by manual data entry.
* **Logic & Transformations:** Applies `F.when(F.col('category').isNull(), None).otherwise(F.initcap('category'))` to column `category`.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** `category` column updated with uniform Title Case formatting.

In [14]:
# Title case fix
df_silver = df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(), None)
     .otherwise(F.initcap("category"))
)

### 📌 Step 13: Verify Standardized Product Categories
* **Purpose:** Confirms that category normalization succeeded and all categories conform to standard taxonomy.
* **Logic & Transformations:** Re-runs `df_silver.select('category').distinct().show()` to verify normalized values.
* **Inputs & Dependencies:** Transformed DataFrame `df_silver`.
* **Outputs & Medallion State:** Distinct sanitized categories logged to cell output.

In [15]:
df_silver.select('category').distinct().show()

+---------+
| category|
+---------+
|Beverages|
|   Snacks|
+---------+



### 📌 Step 14: Correct Typographical Errors in Names and Categories
* **Purpose:** Sanitizes misspelled domain keywords (e.g. replacing erroneous `'Protien'` with `'Protein'`) across product names and categories.
* **Logic & Transformations:** Uses regex replacement `F.regexp_replace(..., '(?i)Protien', 'Protein')` across both `product_name` and `category`.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** `product_name` and `category` fields corrected of common spelling defects.

In [16]:
# Replace 'protien' → 'protein' in both product_name and category
df_silver = (
    df_silver
    .withColumn(
        "product_name",
        F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein")
    )
    .withColumn(
        "category",
        F.regexp_replace(F.col("category"), "(?i)Protien", "Protein")
    )
)


### 📌 Step 15: Preview Intermediate Cleansed Data
* **Purpose:** Visually confirms that text replacements and formatting operations were applied accurately.
* **Logic & Transformations:** Calls `display(df_silver.limit(5))` to render sample cleansed product records.
* **Inputs & Dependencies:** Sanitized DataFrame `df_silver`.
* **Outputs & Medallion State:** Interactive tabular preview in notebook output.

In [17]:
display(df_silver.limit(5))

+----------+---------------------------+---------+-----------------------------------------------------------+--------------------------+---------------+---------+
|product_id|product_name               |category |_metadata                                                  |read_timestamp            |file_name      |file_size|
+----------+---------------------------+---------+-----------------------------------------------------------+--------------------------+---------------+---------+
|P101      |Atlikon Cola 500ml         |Beverages|{sample_data.csv, 1024, s3://spartsbar-2355/products/*.csv}|2026-09-17 12:41:06.195095|sample_data.csv|1024     |
|P102      |Atlikon Orange Juice 1L    |Beverages|{sample_data.csv, 1024, s3://spartsbar-2355/products/*.csv}|2026-09-17 12:41:06.195095|sample_data.csv|1024     |
|P103      |Atlikon Potato Chips 150g  |Snacks   |{sample_data.csv, 1024, s3://spartsbar-2355/products/*.csv}|2026-09-17 12:41:06.195095|sample_data.csv|1024     |
|P104      |Atli

### 📌 Step 16: Business Attribute Extraction & Surrogate Key Generation
* **Purpose:** Derives corporate reporting attributes: categorizes `division`, extracts clean base `product` name and `variant` specification, and generates deterministic SHA-256 surrogate key `product_code`.
* **Logic & Transformations:**
  1. Maps `category` to conformed enterprise `division` (`Nutrition Bars`, `Breakfast Foods`, `Nut Butters & Spreads`).
  2. Extracts packaging specification (e.g. flavor/size) from parentheses via regex into `variant`.
  3. Strips parenthetical text to produce clean base `product` display name.
  4. Generates deterministic surrogate hash: `sha2(trim(product), 256)` as `product_code`.
* **Inputs & Dependencies:** `df_silver` DataFrame.
* **Outputs & Medallion State:** Enriched DataFrame containing `product_code`, `division`, `product`, and `variant`.

In [18]:
### 1: Add division column
df_silver = (
    df_silver
    .withColumn(
        "division",
        F.when(F.col("category") == "Energy Bars",        "Nutrition Bars")
         .when(F.col("category") == "Protein Bars",       "Nutrition Bars")
         .when(F.col("category") == "Granola & Cereals",  "Breakfast Foods")
         .when(F.col("category") == "Recovery Dairy",     "Dairy & Recovery")
         .when(F.col("category") == "Healthy Snacks",     "Healthy Snacks")
         .when(F.col("category") == "Electrolyte Mix",    "Hydration & Electrolytes")
         .otherwise("Other")
    )
)


### 2: Variant column
df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)


### 3: Create new column: product_code  

# Invalid product_ids are replaced with a fallback value to avoid losing fact records and ensure downstream joins remain consistent

df_silver = (
    df_silver
    # 1. Generate deterministic product_code from product_name
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    # 2. Clean product_id: keep only numeric IDs, else set to 999999
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
    # 3. Rename product_name → product
    .withColumnRenamed("product_name", "product")
)

### 📌 Step 17: Select Conformed Silver Master Schema
* **Purpose:** Retains only standardized, conformed columns and audit metadata required by downstream consumers, discarding raw unformatted attributes.
* **Logic & Transformations:** Explicit projection: `product_code`, `division`, `category`, `product`, `variant`, `product_id`, `read_timestamp`, `file_name`, `file_size`.
* **Inputs & Dependencies:** Enriched DataFrame `df_silver`.
* **Outputs & Medallion State:** Conformed Silver DataFrame `df_silver` structured according to enterprise data dictionary.

In [19]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size")

### 📌 Step 18: Preview Conformed Silver Master Products
* **Purpose:** Provides a final interactive sanity check of the cleansed and conformed master dataset before writing to Silver Delta Lake.
* **Logic & Transformations:** Invokes `display(df_silver)` to display the enriched product master in tabular format.
* **Inputs & Dependencies:** Conformed DataFrame `df_silver`.
* **Outputs & Medallion State:** Tabular preview of conformed products rendered in output.

In [20]:
display(df_silver)

+----------------------------------------------------------------+--------+---------+---------------------------+-------+----------+--------------------------+---------------+---------+
|product_code                                                    |division|category |product                    |variant|product_id|read_timestamp            |file_name      |file_size|
+----------------------------------------------------------------+--------+---------+---------------------------+-------+----------+--------------------------+---------------+---------+
|8275a1e4fd6e12bca99731cb197578d0cd2a10e2148705b387d874a1b0485333|Other   |Beverages|Atlikon Cola 500ml         |       |999999    |2026-09-17 12:41:06.195095|sample_data.csv|1024     |
|210621b5f7bfad4cebbb4771290e5fee75125039d1fe5219d57fe2dbb76a8f6a|Other   |Beverages|Atlikon Orange Juice 1L    |       |999999    |2026-09-17 12:41:06.195095|sample_data.csv|1024     |
|8159ce99e70cc060fdb3355d6ccb5a36b53f763c05d586b57c4853cec155a159|Othe

### 📌 Step 19: Persist Master Dataset into Silver Delta Table
* **Purpose:** Commits the sanitized product dimension into the Silver Delta Lake table with Change Data Feed enabled for downstream auditability.
* **Logic & Transformations:** Writes `df_silver` in `overwrite` mode with `mergeSchema=true` and `delta.enableChangeDataFeed=true` to `{catalog}.silver.products`.
* **Inputs & Dependencies:** Conformed DataFrame `df_silver`.
* **Outputs & Medallion State:** Silver Delta table `fmcg.silver.products` saved to storage.

In [21]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

[Local Spark Emulation] Adapted target table: fmcg.silver.products -> silver.products


### 📌 Step 20: Project Attributes for Subsidiary Gold Dimension
* **Purpose:** Prepares the subsidiary product dimension representation containing core reporting attributes and the natural identifier `product_id`.
* **Logic & Transformations:** Queries `fmcg.silver.products` and projects `product_code`, `product_id`, `division`, `category`, `product`, `variant`.
* **Inputs & Dependencies:** Silver Delta table `fmcg.silver.products`.
* **Outputs & Medallion State:** Subsidiary Gold DataFrame `df_gold` ready for persistence.

In [22]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")
df_gold.show(5)

[Local Spark Emulation] Multi-part namespace adapted: SELECT * FROM fmcg.silver.products; -> SELECT * FROM silver.products;
+--------------------+----------+--------+---------+--------------------+-------+
|        product_code|product_id|division| category|             product|variant|
+--------------------+----------+--------+---------+--------------------+-------+
|8275a1e4fd6e12bca...|    999999|   Other|Beverages|  Atlikon Cola 500ml|       |
|210621b5f7bfad4ce...|    999999|   Other|Beverages|Atlikon Orange Ju...|       |
|8159ce99e70cc060f...|    999999|   Other|   Snacks|Atlikon Potato Ch...|       |
|3bdb937f58d10a673...|    999999|   Other|   Snacks|Atlikon Dark Choc...|       |
+--------------------+----------+--------+---------+--------------------+-------+



### 📌 Step 21: Persist Subsidiary Gold Table (`sb_dim_products`)
* **Purpose:** Writes the subsidiary-grain product dimension table `fmcg.gold.sb_dim_products` for subsidiary-level order matching.
* **Logic & Transformations:** Writes `df_gold` with `format('delta')` in `overwrite` mode to `{catalog}.gold.sb_dim_products` with CDF enabled.
* **Inputs & Dependencies:** Subsidiary DataFrame `df_gold`.
* **Outputs & Medallion State:** Gold Delta table `fmcg.gold.sb_dim_products` committed on storage.

In [23]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

[Local Spark Emulation] Adapted target table: fmcg.gold.sb_dim_products -> gold.sb_dim_products


### 📌 Step 22: Prepare Parent Dimension Target & Conformed Source View
* **Purpose:** Instantiates Delta table reference for enterprise parent dimension `dim_products` and prepares conformed deduplicated source view.
* **Logic & Transformations:** Loads `DeltaTable.forName(spark, 'fmcg.gold.dim_products')` and projects conformed distinct attributes (`product_code`, `division`, `category`, `product`, `variant`).
* **Inputs & Dependencies:** Target Delta table `fmcg.gold.dim_products` and Silver products table.
* **Outputs & Medallion State:** Source DataFrame `df_child_products` and target `delta_table` reference in scope.

In [24]:
target_gold_table = f"{catalog}.{gold_schema}.dim_products"
delta_table = DeltaTable.forName(spark, target_gold_table)
df_child_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM {catalog}.{gold_schema}.sb_dim_{data_source};")


26/09/17 12:41:26 WARN CreateNamespaceExec: Namespace gold was created concurrently. Ignoring.


[Local Spark Emulation] DeltaTable.forName adapted: fmcg.gold.dim_products -> gold.dim_products
[Local Spark Emulation] Multi-part namespace adapted: SELECT product_code, division, category, product, variant FROM fmcg.gold.sb_dim_products; -> SELECT product_code, division, category, product, variant FROM gold.sb_dim_products;


### 📌 Step 23: Execute SCD Type 1 Upsert Merge into Parent `dim_products`
* **Purpose:** Synchronizes enterprise product master dimension using Slowly Changing Dimension Type 1 (SCD1) logic, inserting new products and updating modified attributes in place.
* **Logic & Transformations:** Executes Delta Lake `merge()` on condition `target.product_code = source.product_code`. Matching records update `division`, `category`, `product`, and `variant`; non-matching records are inserted as new conformed products.
* **Inputs & Dependencies:** Target Delta table and conformed source DataFrame.
* **Outputs & Medallion State:** Enterprise table `fmcg.gold.dim_products` updated with latest product metadata.

In [25]:
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### 📌 Step 24: Lakehouse Maintenance: File Compaction & Z-ORDER Optimization
* **Purpose:** Eliminates small Parquet files, compacts table storage, and co-locates data along primary query filter keys for high-performance BI queries.
* **Logic & Transformations:** Executes `OPTIMIZE {catalog}.{gold_schema}.dim_products ZORDER BY (product_code)`.
* **Inputs & Dependencies:** Target Delta table `fmcg.gold.dim_products`.
* **Outputs & Medallion State:** Optimized Parquet data layout with multi-dimensional Z-Ordering by `product_code`.

In [26]:
# Maintenance: Compact small files and Z-ORDER by primary join key
spark.sql(f"OPTIMIZE {catalog}.{gold_schema}.dim_products ZORDER BY (product_code)")


[Local Spark Emulation] Multi-part namespace adapted: OPTIMIZE fmcg.gold.dim_products ZORDER BY (product_code) -> OPTIMIZE gold.dim_products ZORDER BY (product_code)


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,